# 02 - CSV → Delta Lake (Bronze)

Lê os arquivos CSV do bucket `landing-zone` e converte para **Delta Lake** no bucket `bronze`.

```
MinIO / landing-zone / <tabela>.csv  →  MinIO / bronze / <tabela>  (Delta Table)
```

> Execute o notebook `01_sqlserver_to_minio.ipynb` antes deste.


In [ ]:
import os
import sys
import boto3
from botocore.exceptions import ClientError
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"]           = "C:\\hadoop"
os.environ["JAVA_TOOL_OPTIONS"]     = "-Djava.library.path=C:\\hadoop\\bin"

MINIO_ENDPOINT = "http://localhost:9020"
MINIO_ACCESS   = "minioadmin"
MINIO_SECRET   = "minioadmin"
LANDING        = "s3a://landing-zone"
BRONZE         = "s3a://bronze"

In [ ]:
# Inicia o Spark com Delta Lake e suporte ao MinIO via S3A
builder = (
    SparkSession.builder
    .appName("CSV para Delta Lake - MinIO")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",              MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",            MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",            MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access",     "true")
    .config("spark.hadoop.fs.s3a.impl",                  "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "2g")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark {spark.version} iniciado com Delta Lake + MinIO S3A")

In [ ]:
# Cria o bucket bronze no MinIO (se não existir)
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET,
)

try:
    s3.create_bucket(Bucket="bronze")
    print("Bucket 'bronze' criado.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print("Bucket 'bronze' já existia.")
    else:
        raise

In [ ]:
# Lê cada CSV do landing-zone e salva como Delta no bronze
tabelas = ["clientes", "produtos", "pedidos"]

for tabela in tabelas:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{LANDING}/{tabela}/{tabela}.csv")
    )

    df.write.format("delta").mode("overwrite").save(f"{BRONZE}/{tabela}")
    print(f"\n=== {tabela.upper()} salvo em Delta ({df.count()} registros) ===")
    df.show(truncate=False)

In [ ]:
# Detalha as tabelas Delta criadas no bronze
from delta.tables import DeltaTable

for tabela in tabelas:
    dt = DeltaTable.forPath(spark, f"{BRONZE}/{tabela}")
    print(f"\n--- Detalhes: {tabela} ---")
    dt.detail().select("name", "format", "numFiles", "sizeInBytes").show(truncate=False)
    print("Histórico:")
    dt.history().select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
spark.stop()
print("Sessão Spark encerrada.")